# Lab 2 · Hyperparameter tuning with nested runs
**Block 1 · about 30 min.** One **parent run** for the search, one **child run** per trial.
The parent keeps the winner; the children keep the evidence.

In [ ]:
import sys, warnings
sys.path.insert(0, "../src")          # churn_data.py, churn_model.py
warnings.filterwarnings("ignore")

import mlflow
mlflow.set_tracking_uri("http://127.0.0.1:5000")   # the server you started in the terminal
mlflow.set_experiment("churn-workshop")
print("MLflow", mlflow.__version__)

In [ ]:
import optuna
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from churn_data import history, time_split
from churn_model import make_pipeline

optuna.logging.set_verbosity(optuna.logging.WARNING)
X_tr, y_tr, X_va, y_va = time_split(history())

## Step 1 · Define one trial = one child run

In [ ]:
def objective(trial):
    params = {
        "max_depth": trial.suggest_int("max_depth", 2, 8),
        "n_estimators": trial.suggest_int("n_estimators", 100, 500, step=100),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
    }
    with mlflow.start_run(run_name=f"trial_{trial.number}", nested=True):
        model = make_pipeline(XGBClassifier(**params, eval_metric="logloss")).fit(X_tr, y_tr)
        auc = roc_auc_score(y_va, model.predict_proba(X_va)[:, 1])
        mlflow.log_params(params)
        mlflow.log_metric("val_auc", auc)
    return auc

## Step 2 · Run the search inside a parent run

In [ ]:
with mlflow.start_run(run_name="xgb_optuna_search") as parent:
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=0))
    study.optimize(objective, n_trials=15)

    mlflow.log_params({f"best_{k}": v for k, v in study.best_params.items()})
    mlflow.log_metric("best_val_auc", study.best_value)
    mlflow.set_tag("search", "optuna-tpe")

print("best AUC:", round(study.best_value, 4), study.best_params)

👉 **In the UI:** expand the parent run to see its 15 children. Select them all → **Compare** →
*Parallel coordinates* chart. Which hyperparameter matters most?

## Step 3 · Find the best child from code
Child runs carry the tag `mlflow.parentRunId`. We'll reuse these parameters in Lab 3.

In [ ]:
best = mlflow.search_runs(
    experiment_names=["churn-workshop"],
    filter_string=f"tags.mlflow.parentRunId = '{parent.info.run_id}'",
    order_by=["metrics.val_auc DESC"], max_results=1).iloc[0]
best_params = {k.replace("params.", ""): v for k, v in best.items() if k.startswith("params.")}
print(best["tags.mlflow.runName"], round(best["metrics.val_auc"], 4), best_params)

## ✍️ Your turn (5 min)
Params come back from `search_runs` as **strings**. Convert `best_params` to the right types
(`int` for depth and estimators, `float` for the rest) so you can pass them to `XGBClassifier`.
Why do you think MLflow stores parameters as strings?